# 🔬 Prueba Piloto: Generación y Evaluación

Diseño y análisis de una prueba piloto de campo para validar el modelo de detección de fraude.

**Flujo completo:**

**Fase 1 — Generación:**
1. Cargar predicciones de inferencia
2. Por cada región, seleccionar los **top N clientes con mayor probabilidad** (grupo modelo)
3. Por cada región, seleccionar **N clientes al azar** (grupo control)
4. Exportar dos CSVs listos para que las cuadrillas registren el resultado de la inspección

**Fase 2 — Evaluación post-inspección:**
5. Cargar los CSVs con la columna `fraude_real` completada por los inspectores
6. Calcular métricas: precisión, tasa de detección, lift sobre el azar
7. Comparar estadísticamente el modelo vs selección aleatoria
8. Conclusión: ¿el modelo prioriza mejor que el azar?

**¿Por qué un grupo de control aleatorio?**
Sin un grupo de control, no podés saber si el modelo es mejor que simplemente inspeccionar clientes al azar. Si el 5% de los clientes aleatorios son fraude y el 25% de los del modelo son fraude, el modelo tiene un **lift de 5x**.

## 0. Configuración

In [ ]:
# ============================================================
# CONFIGURACIÓN — paths, archivos y parámetros
# ============================================================
# Toda la configuración está acá. Para apuntar la notebook a otro
# proyecto o dataset, modificá estos valores (no hace falta tocar el
# resto de las celdas).

from pathlib import Path

# --- Proyecto y datos ---
PROJECT_PATH = Path('<<PROJECT_PATH>>')
OUTPUT_PATH = PROJECT_PATH / 'output'
VERSION = '<<VERSION>>'
INFERENCE_DIR = '<<INFERENCE_DIR>>'

# --- Predicciones para la prueba piloto ---
# Usamos las predicciones ETIQUETADAS (v4) porque incluyen la columna target
# (outcome real), necesaria para la Fase 2 de evaluación (comparar modelo vs
# aleatorio con outcomes reales). También tienen probability y geo_region.
PREDICTIONS_CSV = OUTPUT_PATH / VERSION / INFERENCE_DIR / 'predictions.csv'

# --- Entorno (detección automática Colab vs local) ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# === PARÁMETROS DE LA PRUEBA PILOTO ===

# Cuántos clientes inspeccionar POR REGIÓN en cada grupo
N_PER_REGION = 100

# Columna que define las regiones (debe existir en el CSV)
REGION_COL = "geo_region"

# Semilla para reproducibilidad del muestreo aleatorio
RANDOM_SEED = 42

# === SALIDA (FASE 1) ===
OUTPUT_DIR = OUTPUT_PATH / VERSION / 'pilot_test'

# === ENTRADA POST-INSPECCIÓN (FASE 2) ===
# Después de que las cuadrillas completen las inspecciones, actualizá estos paths
INSPECTED_MODEL_CSV = OUTPUT_DIR / 'inspecciones_modelo_resultados.csv'
INSPECTED_RANDOM_CSV = OUTPUT_DIR / 'inspecciones_aleatorio_resultados.csv'

print(f'PROJECT_PATH     : {PROJECT_PATH}')
print(f'PREDICTIONS_CSV  : {PREDICTIONS_CSV}')
print(f'OUTPUT_DIR       : {OUTPUT_DIR}')
print(f'IN_COLAB         : {IN_COLAB}')

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, fisher_exact

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 6)

assert Path(PREDICTIONS_CSV).exists(), f"No se encuentra: {PREDICTIONS_CSV}"
print("✅ Configuración lista")

---
# FASE 1 — GENERACIÓN DE LISTAS DE INSPECCIÓN

## 1. Cargar predicciones y explorar regiones

Cargamos el CSV de predicciones generado por `energizados run infer`. Este archivo contiene todos los clientes con su `probability` de fraude y las columnas de features (incluyendo región si se usó `GeoFeaturesETL`).

**Qué revisamos en esta sección:**
- Cantidad total de clientes disponibles para muestrear
- Qué regiones (`REGION_COL`) existen en los datos y cuántas son
- Que la columna de región esté presente (error temprano si falta)
- Cuántos clientes por región — para saber si hay suficientes para el muestreo 2N

> **IMPORTANTE:** La variable `N_PER_REGION` define cuántos clientes seleccionar por grupo en cada región. Elegila según la capacidad operativa de tus cuadrillas de inspección.

In [ ]:
df = pd.read_csv(PREDICTIONS_CSV)
print(f"Total clientes: {len(df):,}")
print(f"Columnas: {list(df.columns)}")

# Verificar que existe la columna de región
if REGION_COL not in df.columns:
    raise ValueError(f"Columna '{REGION_COL}' no encontrada. Disponibles: {list(df.columns)}")

# Regiones disponibles
regions = sorted(df[REGION_COL].dropna().unique())
print(f"\nRegiones ({len(regions)}): {regions}")

# Clientes por región
region_counts = df[REGION_COL].value_counts()
print(f"\nClientes por región:")
for region, count in region_counts.items():
    flag = "⚠️  < N" if count < N_PER_REGION * 2 else ""
    print(f"  {region}: {count:,} {flag}")

## 2. Armar la selección por región

Para cada región seleccionamos dos grupos **sin solapamiento**:
- **Grupo Modelo:** top N clientes con mayor `probability` (los más sospechosos según el modelo).
- **Grupo Aleatorio:** N clientes elegidos al azar de entre los **restantes** (excluyendo los ya seleccionados por el modelo, para evitar medir dos veces al mismo cliente).

**Edge cases:**
- Si una región tiene menos de 2N clientes, ajustamos automáticamente.
- Si la región tiene probabilidades empatadas en el borde del top N, incluimos todos los empatados (ligeramente más de N).

In [ ]:
model_rows = []
random_rows = []
summary_rows = []

for region in regions:
    region_df = df[df[REGION_COL] == region].copy()
    n_total = len(region_df)
    
    # Ajustar N si la región es chica
    n_effective = min(N_PER_REGION, n_total // 2)
    if n_effective < N_PER_REGION:
        print(f"⚠️  {region}: solo {n_total} clientes → N ajustado a {n_effective}")
    
    if n_effective == 0:
        print(f"⚠️  {region}: muy pocos clientes ({n_total}), saltando")
        continue
    
    # Top N por probabilidad (modelo)
    top_n = region_df.nlargest(n_effective, "probability")
    top_n["grupo"] = "modelo"
    model_rows.append(top_n)
    
    # Aleatorio entre los que NO están en el top N
    remaining = region_df.drop(top_n.index)
    n_random = min(n_effective, len(remaining))
    random_n = remaining.sample(n=n_random, random_state=RANDOM_SEED)
    random_n["grupo"] = "aleatorio"
    random_rows.append(random_n)
    
    # Stats para el resumen
    summary_rows.append({
        "region": region,
        "n_total": n_total,
        "n_modelo": len(top_n),
        "n_aleatorio": len(random_n),
        "prob_media_modelo": top_n["probability"].mean(),
        "prob_media_aleatorio": random_n["probability"].mean(),
        "prob_min_modelo": top_n["probability"].min(),
        "prob_max_aleatorio": random_n["probability"].max(),
    })

# Consolidar
df_modelo = pd.concat(model_rows, ignore_index=True)
df_aleatorio = pd.concat(random_rows, ignore_index=True)
df_summary = pd.DataFrame(summary_rows)

print(f"\n=== RESUMEN DE LA SELECCIÓN ===")
print(f"Grupo Modelo:    {len(df_modelo):,} clientes")
print(f"Grupo Aleatorio: {len(df_aleatorio):,} clientes")
print(f"Total a inspeccionar: {len(df_modelo) + len(df_aleatorio):,}")

# Verificar que no hay solapamiento
overlap = set(df_modelo["cliente"]) & set(df_aleatorio["cliente"])
if overlap:
    print(f"⚠️  Hay {len(overlap)} clientes en ambos grupos — esto no debería pasar")
else:
    print("✅ Sin solapamiento entre grupos")

display(df_summary)

## 3. Visualizar la selección

Comparamos las distribuciones de probabilidad de ambos grupos para verificar que el grupo modelo efectivamente tiene probabilidades más altas que el aleatorio.

**Qué esperamos ver:**
- **Boxplot izquierdo:** el grupo Modelo (rojo) debería tener la mediana claramente más alta que el grupo Aleatorio (azul). Si se solapan mucho, el modelo no está discriminando bien en esa región.
- **Histograma derecho:** el grupo Modelo debería estar concentrado en probabilidades altas (> 0.7), mientras que el Aleatorio debería reflejar la distribución natural de la población.

Si el modelo no logra separarse del azar en esta visualización, la prueba piloto probablemente no mostrará diferencias significativas en campo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplots comparativos
combined = pd.concat([
    df_modelo[["probability", REGION_COL]].assign(grupo="Modelo"),
    df_aleatorio[["probability", REGION_COL]].assign(grupo="Aleatorio"),
])
sns.boxplot(data=combined, x="grupo", y="probability", palette={"Modelo": "#F44336", "Aleatorio": "#2196F3"}, ax=axes[0])
axes[0].set_title("Distribución de probabilidad por grupo")
axes[0].set_ylabel("Probability")

# Por región
region_comparison = df_summary.melt(
    id_vars=["region"],
    value_vars=["prob_media_modelo", "prob_media_aleatorio"],
    var_name="grupo", value_name="prob_media"
)
region_comparison["grupo"] = region_comparison["grupo"].map({
    "prob_media_modelo": "Modelo", "prob_media_aleatorio": "Aleatorio"
})

sns.barplot(data=region_comparison, x="region", y="prob_media", hue="grupo",
            palette={"Modelo": "#F44336", "Aleatorio": "#2196F3"}, ax=axes[1])
axes[1].set_title("Probabilidad media por región y grupo")
axes[1].set_ylabel("Probabilidad media")
axes[1].set_xlabel("Región")
plt.xticks(rotation=45, ha="right")

plt.suptitle("Validación de la selección:Modelo vs Aleatorio", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Preparar CSVs para las cuadrillas

Agregamos columnas para que los inspectores registren el resultado de cada visita.

**Columnas que debe llenar el inspector:**
- `fraude_real` → `1` si se confirmó fraude, `0` si el cliente es legítimo, dejar vacío si no se pudo inspeccionar.
- `fecha_inspeccion` → fecha en que se realizó la visita (formato `YYYY-MM-DD`).
- `inspector` → nombre o ID del inspector.
- `observaciones` → notas de campo: qué se encontró, tipo de irregularidad, etc.

**Columnas de apoyo para identificar al cliente:**
- `cliente`, `geo_region`, `probability` (para priorizar dentro de la lista)
- Consumos de los últimos meses (para que el inspector pueda contrastar en campo)

In [ ]:
# Columnas de consumo para incluir
CONSUMPTION_COLS = [c for c in df.columns if c.endswith("_anterior")]

# Columnas base
base_cols = ["cliente", REGION_COL, "probability"]
base_cols += [c for c in CONSUMPTION_COLS if c in df.columns]

# Columnas para el inspector
inspection_cols = ["fraude_real", "fecha_inspeccion", "inspector", "observaciones"]

def prepare_inspection_csv(df_group, group_name):
    """Prepara un CSV listo para imprimir/entregar a cuadrillas."""
    out = df_group[base_cols].copy()
    out = out.sort_values([REGION_COL, "probability"], ascending=[True, False])
    # Agregar columnas vacías para que el inspector complete
    for col in inspection_cols:
        out[col] = ""
    return out

df_modelo_out = prepare_inspection_csv(df_modelo, "modelo")
df_aleatorio_out = prepare_inspection_csv(df_aleatorio, "aleatorio")

print(f"CSV Modelo:    {len(df_modelo_out)} filas × {len(df_modelo_out.columns)} columnas")
print(f"CSV Aleatorio: {len(df_aleatorio_out)} filas × {len(df_aleatorio_out.columns)} columnas")
print(f"\nColumnas: {list(df_modelo_out.columns)}")
df_modelo_out.head()

In [ ]:
# Exportar CSVs
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

modelo_path = output_dir / "inspecciones_modelo.csv"
aleatorio_path = output_dir / "inspecciones_aleatorio.csv"

df_modelo_out.to_csv(modelo_path, index=False)
df_aleatorio_out.to_csv(aleatorio_path, index=False)

print(f"✅ {modelo_path}")
print(f"✅ {aleatorio_path}")

# Exportar también un resumen de la prueba para documentar
summary_path = output_dir / "resumen_prueba_piloto.csv"
df_summary.to_csv(summary_path, index=False)
print(f"✅ {summary_path}")

print(f"\n📋 Instrucciones para las cuadrillas:")
print(f"   1. Entregar {modelo_path.name} a la mitad de las cuadrillas")
print(f"   2. Entregar {aleatorio_path.name} a la otra mitad")
print(f"   3. NO decirles cuál es cuál (evita sesgo del inspector)")
print(f"   4. Cada cliente visitado: completar fraude_real (1=SÍ, 0=NO)")
print(f"   5. Si no se pudo inspeccionar: dejar fraude_real vacío")

---
# FASE 2 — EVALUACIÓN POST-INSPECCIÓN

> ⚠️ Ejecutá esta sección **después** de que las cuadrillas hayan completado las inspecciones y hayas guardado los CSVs con la columna `fraude_real` llena.

## 5. Cargar resultados de inspección

Cargamos los CSVs que las cuadrillas devolvieron con `fraude_real` completado.

**Validaciones:**
- `fraude_real` solo puede ser 0, 1, o vacío (no inspeccionado).
- Clientes sin inspeccionar se excluyen del análisis de métricas pero se reportan.

In [ ]:
def load_inspected(path, grupo_label):
    """Carga un CSV de inspección y valida los datos."""
    p = Path(path)
    if not p.exists():
        print(f"⚠️  {p} no existe todavía. Ejecutá primero la Fase 1 y completá las inspecciones.")
        return None
    
    df_insp = pd.read_csv(p)
    print(f"{grupo_label}: {len(df_insp)} clientes cargados")
    
    # Validar fraude_real
    if "fraude_real" not in df_insp.columns:
        print(f"  ⚠️  Columna 'fraude_real' no encontrada. ¿Completaron las inspecciones?")
        return None
    
    # Convertir a numérico, coercing errors
    df_insp["fraude_real"] = pd.to_numeric(df_insp["fraude_real"], errors="coerce")
    
    inspected = df_insp[df_insp["fraude_real"].notna()]
    not_inspected = df_insp[df_insp["fraude_real"].isna()]
    
    print(f"  Inspeccionados:     {len(inspected)} ({len(inspected)/len(df_insp)*100:.1f}%)")
    print(f"  No inspeccionados:  {len(not_inspected)} ({len(not_inspected)/len(df_insp)*100:.1f}%)")
    
    # Validar valores
    valid_vals = inspected["fraude_real"].isin([0, 1])
    if not valid_vals.all():
        bad = inspected[~valid_vals]
        print(f"  ⚠️  {len(bad)} filas con fraude_real != 0/1 (se excluyen)")
        inspected = inspected[valid_vals]
    
    fraud_count = int(inspected["fraude_real"].sum())
    legit_count = len(inspected) - fraud_count
    print(f"  Fraudes confirmados: {fraud_count} ({fraud_count/len(inspected)*100:.1f}%)")
    print(f"  Clientes legítimos:  {legit_count}")
    
    return inspected

print("=== CARGA DE RESULTADOS ===\n")
df_modelo_res = load_inspected(INSPECTED_MODEL_CSV, "Modelo")
df_aleatorio_res = load_inspected(INSPECTED_RANDOM_CSV, "Aleatorio")

## 6. Métricas comparativas

Comparamos el rendimiento del modelo vs la selección aleatoria.

**Métricas clave:**
- **Precisión (precision):** de los clientes que inspeccionaste, ¿cuántos eran realmente fraude? `fraudes_detectados / total_inspeccionados`
- **Tasa de detección:** fraudes encontrados por cada 100 inspecciones.
- **Lift:** ¿cuántas veces más fraude encuentra el modelo que el azar? `precision_modelo / precision_aleatorio`
- **Diferencia absoluta:** `precision_modelo - precision_aleatorio`

**Interpretación del lift:**
- Lift = 1 → el modelo no es mejor que tirar una moneda.
- Lift = 2 → el modelo encuentra el doble de fraudes por inspección.
- Lift = 5 → el modelo es 5 veces más eficiente que inspeccionar al azar.
- Lift < 1 → el modelo es **peor** que el azar (problema grave).

In [ ]:
if df_modelo_res is not None and df_aleatorio_res is not None:
    # Métricas
    n_modelo = len(df_modelo_res)
    n_random = len(df_aleatorio_res)
    fraudes_modelo = int(df_modelo_res["fraude_real"].sum())
    fraudes_random = int(df_aleatorio_res["fraude_real"].sum())
    
    prec_modelo = fraudes_modelo / n_modelo
    prec_random = fraudes_random / n_random
    lift = prec_modelo / prec_random if prec_random > 0 else float("inf")
    diff = prec_modelo - prec_random
    
    print("=" * 60)
    print("RESULTADOS DE LA PRUEBA PILOTO")
    print("=" * 60)
    print(f"{'':<30} {'Modelo':>12} {'Aleatorio':>12}")
    print(f"{'Inspeccionados':<30} {n_modelo:>12,} {n_random:>12,}")
    print(f"{'Fraudes detectados':<30} {fraudes_modelo:>12,} {fraudes_random:>12,}")
    print(f"{'Precisión (precision)':<30} {prec_modelo:>11.2%} {prec_random:>11.2%}")
    print(f"{'Tasa detección (×100 inspec.)':<30} {prec_modelo*100:>11.1f} {prec_random*100:>11.1f}")
    print()
    print(f"Lift del modelo sobre el azar: {lift:.2f}x")
    print(f"Diferencia absoluta:         {diff:+.2%}")
    print("=" * 60)
else:
    print("⚠️  Faltan datos de inspección. Completá la Fase 1, hacé las inspecciones, y volvé a correr.")

## 7. Test estadístico: ¿la diferencia es significativa?

No alcanza con que el modelo tenga mejor precisión — necesitamos saber si la diferencia es estadísticamente significativa o puede deberse al azar.

Usamos el **test exacto de Fisher**, apropiado para muestras chicas (tamaños de prueba piloto típicos).

**Hipótesis:**
- H₀ (nula): No hay diferencia entre el modelo y el azar. La precisión es la misma.
- H₁ (alternativa): El modelo es mejor que el azar (más fraudes detectados por inspección).

**Cómo leer el p-valor:**
- `p < 0.05` → diferencia **estadísticamente significativa**. Podemos confiar en que el modelo es mejor.
- `p < 0.01` → diferencia **altamente significativa**.
- `p >= 0.05` → no podemos descartar que la diferencia sea producto del azar. Necesitamos más datos (más inspecciones).

In [ ]:
if df_modelo_res is not None and df_aleatorio_res is not None:
    n_modelo = len(df_modelo_res)
    n_random = len(df_aleatorio_res)
    fraudes_modelo = int(df_modelo_res["fraude_real"].sum())
    fraudes_random = int(df_aleatorio_res["fraude_real"].sum())
    
    # Tabla de contingencia para Fisher
    #              | Fraude | No Fraude |
    # Modelo        |   a    |    b      |
    # Aleatorio     |   c    |    d      |
    table = [[
        fraudes_modelo, n_modelo - fraudes_modelo
    ], [
        fraudes_random, n_random - fraudes_random
    ]]
    
    # Fisher exact test (two-sided por defecto, pero usamos one-sided "greater"
    # porque la hipótesis es que modelo > aleatorio)
    odds_ratio, p_value = fisher_exact(table, alternative="greater")
    
    print("=== TEST ESTADÍSTICO ===")
    print(f"Tabla de contingencia:")
    print(f"                  Fraude  No Fraude")
    print(f"  Modelo          {fraudes_modelo:>6}  {n_modelo - fraudes_modelo:>9}")
    print(f"  Aleatorio       {fraudes_random:>6}  {n_random - fraudes_random:>9}")
    print()
    print(f"Odds ratio: {odds_ratio:.4f}")
    print(f"P-valor (one-sided, modelo > aleatorio): {p_value:.6f}")
    print()
    
    if p_value < 0.01:
        print("✅ Diferencia ALTAMENTE SIGNIFICATIVA (p < 0.01)")
        print("   El modelo es claramente mejor que el azar.")
    elif p_value < 0.05:
        print("✅ Diferencia SIGNIFICATIVA (p < 0.05)")
        print("   Podemos confiar en que el modelo prioriza mejor.")
    else:
        print(f"⚠️  Diferencia NO significativa (p = {p_value:.4f})")
        print("   No hay evidencia suficiente. Posibles causas:")
        print("   - Muestra muy chica (aumentar N_PER_REGION)")
        print("   - El modelo realmente no es mejor que el azar")
        print("   - Las inspecciones no se completaron correctamente")
else:
    print("⚠️  Faltan datos.")

## 8. Desglose por región

¿El modelo funciona mejor en algunas regiones que en otras?

In [ ]:
if df_modelo_res is not None and df_aleatorio_res is not None:
    region_metrics = []
    
    for region in sorted(set(df_modelo_res[REGION_COL].unique()) | set(df_aleatorio_res[REGION_COL].unique())):
        m = df_modelo_res[df_modelo_res[REGION_COL] == region]
        r = df_aleatorio_res[df_aleatorio_res[REGION_COL] == region]
        
        if len(m) == 0 or len(r) == 0:
            continue
        
        prec_m = m["fraude_real"].mean()
        prec_r = r["fraude_real"].mean()
        
        region_metrics.append({
            "region": region,
            "n_modelo": len(m),
            "fraudes_modelo": int(m["fraude_real"].sum()),
            "prec_modelo": prec_m,
            "n_aleatorio": len(r),
            "fraudes_aleatorio": int(r["fraude_real"].sum()),
            "prec_aleatorio": prec_r,
            "lift": prec_m / prec_r if prec_r > 0 else float("inf"),
        })
    
    region_df = pd.DataFrame(region_metrics)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(region_df) * 0.5)))
    
    # Barras agrupadas
    x = np.arange(len(region_df))
    w = 0.35
    axes[0].bar(x - w/2, region_df["prec_modelo"] * 100, w, color="#F44336", label="Modelo")
    axes[0].bar(x + w/2, region_df["prec_aleatorio"] * 100, w, color="#2196F3", label="Aleatorio")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(region_df["region"], rotation=45, ha="right")
    axes[0].set_ylabel("Precisión (%)")
    axes[0].set_title("Precisión por región: Modelo vs Aleatorio")
    axes[0].legend()
    
    # Lift por región
    lifts = region_df["lift"].clip(upper=10)  # cap para visualización
    colors = ["#4CAF50" if l >= 1 else "#F44336" for l in region_df["lift"]]
    axes[1].barh(region_df["region"], lifts, color=colors, edgecolor="white")
    axes[1].axvline(1, color="gray", linestyle="--", linewidth=1.5, label="Lift=1 (azar)")
    axes[1].set_xlabel("Lift (× sobre el azar)")
    axes[1].set_title("Lift del modelo por región")
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    display(region_df.style.format({
        "prec_modelo": "{:.1%}", "prec_aleatorio": "{:.1%}", "lift": "{:.2f}x"
    }))
else:
    print("⚠️  Faltan datos.")

## 9. Curva de eficiencia: fraudes detectados vs inspecciones

Simulamos cuántos fraudes habríamos detectado si inspeccionábamos en orden de probabilidad (modelo) vs orden aleatorio.

**Cómo leerlo:**
- La curva del modelo debería **subir más rápido** que la del azar: encuentra más fraudes con menos inspecciones.
- El área entre las dos curvas representa la **ganancia** de usar el modelo.
- Si las curvas son casi iguales, el modelo no está priorizando mejor que el azar.

In [ ]:
if df_modelo_res is not None and df_aleatorio_res is not None:
    # Ordenar por probabilidad descendente
    modelo_sorted = df_modelo_res.sort_values("probability", ascending=False)
    random_sorted = df_aleatorio_res.sample(frac=1, random_state=RANDOM_SEED)  # shuffle
    
    # Fraudes acumulados
    modelo_cumsum = modelo_sorted["fraude_real"].cumsum()
    random_cumsum = random_sorted["fraude_real"].cumsum()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x_modelo = range(1, len(modelo_cumsum) + 1)
    x_random = range(1, len(random_cumsum) + 1)
    
    ax.plot(x_modelo, modelo_cumsum, color="#F44336", linewidth=2.5, label=f"Modelo (total={modelo_cumsum.iloc[-1]:.0f})")
    ax.plot(x_random, random_cumsum, color="#2196F3", linewidth=2, linestyle="--", label=f"Aleatorio (total={random_cumsum.iloc[-1]:.0f})")
    
    # Línea de referencia: si todos fueran fraude (cota superior)
    ax.plot([0, max(len(x_modelo), len(x_random))], [0, max(len(x_modelo), len(x_random))],
            "gray", alpha=0.3, linewidth=0.8, label="Cota superior (100% precisión)")
    
    ax.set_xlabel("Inspecciones realizadas")
    ax.set_ylabel("Fraudes detectados (acumulado)")
    ax.set_title("Eficiencia: fraudes detectados vs inspecciones realizadas")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Faltan datos.")

## 10. Conclusión y recomendaciones

Sintetizamos los hallazgos de la prueba piloto en una recomendación accionable.

**Qué responde esta sección:**
- ¿El modelo detecta más fraudes que una selección aleatoria?
- ¿Cuál es el **lift** (cuántas veces mejor es el modelo)?
- ¿El resultado es estadísticamente significativo o puede deberse al azar?
- ¿Vale la pena adoptar el modelo para priorizar inspecciones?

**Criterios de decisión:**
- **Lift > 1.5 y p < 0.05:** el modelo es claramente mejor. Recomendación: **adoptar**.
- **Lift > 1.2 y p < 0.1:** hay una mejora pero es marginal. Recomendación: **evaluar con más datos** antes de decidir.
- **Lift < 1.2 o p > 0.1:** la diferencia no es concluyente. Posibles causas: muestra muy chica, modelo no generaliza a esta población, o el fraude es más aleatorio de lo esperado.

> **Para el reporte ejecutivo:** incluí el lift, el p-valor del test de Fisher, y el ahorro estimado en inspecciones (usando el costo operativo por inspección).

In [ ]:
if df_modelo_res is not None and df_aleatorio_res is not None:
    fraudes_modelo = int(df_modelo_res["fraude_real"].sum())
    fraudes_random = int(df_aleatorio_res["fraude_real"].sum())
    n_modelo = len(df_modelo_res)
    n_random = len(df_aleatorio_res)
    prec_modelo = fraudes_modelo / n_modelo
    prec_random = fraudes_random / n_random
    lift = prec_modelo / prec_random if prec_random > 0 else float("inf")
    
    print("=" * 60)
    print("CONCLUSIÓN DE LA PRUEBA PILOTO")
    print("=" * 60)
    print()
    print(f"Se inspeccionaron {n_modelo + n_random:,} clientes en total.")
    print(f"  Grupo Modelo:    {n_modelo:,} inspecciones → {fraudes_modelo} fraudes ({prec_modelo:.1%})")
    print(f"  Grupo Aleatorio: {n_random:,} inspecciones → {fraudes_random} fraudes ({prec_random:.1%})")
    print()
    
    if lift >= 2 and p_value < 0.05:
        print(f"✅ El modelo es {lift:.1f}x más efectivo que el azar (p={p_value:.4f}).")
        print("   Recomendación: ADOPTAR el modelo para priorizar inspecciones.")
        print(f"   Impacto estimado: por cada 100 inspecciones, el modelo encuentra")
        print(f"   {int((prec_modelo - prec_random) * 100)} fraudes más que el azar.")
    elif lift >= 2 and p_value >= 0.05:
        print(f"⚠️  El modelo muestra {lift:.1f}x de lift pero NO es estadísticamente significativo (p={p_value:.4f}).")
        print("   Recomendación: AMPLIAR la prueba piloto con más clientes para confirmar.")
    elif lift >= 1.2:
        print(f"🔍 El modelo tiene un lift modesto de {lift:.1f}x (p={p_value:.4f}).")
        print("   Recomendación: el modelo aporta, pero considerar si el costo de mantenimiento lo justifica.")
    else:
        print(f"🔴 El modelo NO muestra mejora sobre el azar (lift={lift:.2f}x, p={p_value:.4f}).")
        print("   Recomendación: REVISAR el modelo — posible sobreajuste, data leakage, o features no generalizables.")
    
    print()
    print("Próximos pasos sugeridos:")
    print("  1. Revisar los SHAP values para entender qué features usa el modelo")
    print("  2. Analizar si el modelo falla en regiones o perfiles específicos")
    print("  3. Reentrenar con datos más representativos si es necesario")
    print("=" * 60)
else:
    print("⚠️  Completá las inspecciones primero (Fase 1 → campo → Fase 2).")

---
## Notas operativas

- **Ciego simple:** idealmente las cuadrillas no deberían saber qué grupo están inspeccionando (para evitar sesgo de confirmación).
- **Calidad de datos:** `fraude_real` debe ser determinado con criterio objetivo (evidencia de manipulación del medidor, bypass, etc.), no con opinión del inspector.
- **No inspeccionados:** si muchos clientes no pudieron ser inspeccionados (> 20%), la prueba pierde validez. Registrar el motivo.
- **Tamaño de muestra:** N=100 por región es un buen punto de partida. Si necesitás más poder estadístico, aumentalo.
- **Ética:** algunos clientes del grupo aleatorio pueden ser fraude real. La prueba piloto tiene un costo ético (fraudes no detectados a propósito). Considerá si tu organización lo aprueba.